# Step32 — Spatial Validation

## Main objective

Spatially validate the malignant-cell biological framework frozen in
Steps 23–31, with primary emphasis on the conserved MS3
proliferative/mitotic high-risk program and secondary emphasis on the
MS6 differentiated/hepatic/metabolic counter-pole.

## Frozen biological targets

Primary:
- Step25 frozen MS3 Conserved Core50
- MS3 proliferative / G2M / mitotic / spindle-chromosome biology

Secondary:
- MS6 hepatic differentiation / bile-acid / metabolic biology
- Reciprocal MS3-like versus MS6-like spatial organization

Mechanistic context:
- Frozen Step29 regulatory architecture
- Selected Step30 signaling architecture only when spatial data support it

## Firewall

Step32 will NOT:
- reselect Core50 genes;
- reconstruct the Original RiskScore;
- redefine MS1–MS8 states;
- discover new malignant states;
- optimize a new spatial signature;
- rerun unbiased GRN or communication discovery.

Step32A performs data/platform audit only.

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step32A Cell 244
# Environment and project initialization
# =============================================================================

from pathlib import Path
import os
import sys
import json
import re
import platform
import pandas as pd
import numpy as np

try:
    import scanpy as sc
except Exception:
    sc = None

try:
    import anndata as ad
except Exception:
    ad = None


TCGA_ROOT = Path(_hcc_path(''))

STEP32_ROOT = (
    TCGA_ROOT /
    "LIHC_scRNA_Step32_Spatial_Validation"
)

DATA_DIR = STEP32_ROOT / "Data"
TABLE_DIR = STEP32_ROOT / "Tables"
FIGURE_DIR = STEP32_ROOT / "Figures"
AUDIT_DIR = STEP32_ROOT / "Audit"
RESOURCE_DIR = STEP32_ROOT / "Resources"


for d in [
    STEP32_ROOT,
    DATA_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    AUDIT_DIR,
    RESOURCE_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )


print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Executable:", sys.executable)

print(
    "Scanpy:",
    None if sc is None else sc.__version__
)

print(
    "AnnData:",
    None if ad is None else ad.__version__
)

print("\nTCGA root:", TCGA_ROOT)
print("TCGA root exists:", TCGA_ROOT.exists())

print("\nStep32 root:", STEP32_ROOT)
print("Step32 root exists:", STEP32_ROOT.exists())

In [ ]:
# =============================================================================
# Step32A Cell 245
# Read-only filesystem audit for spatial-transcriptomics candidates
# =============================================================================

SPATIAL_KEYWORDS = [
    "spatial",
    "visium",
    "slide",
    "stereo",
    "stereoseq",
    "geomx",
    "xenium",
    "cosmx",
    "merfish",
    "seqscope",
    "hdst",
    "tissue_positions",
    "scalefactors",
    "hires_image",
    "lowres_image"
]

RELEVANT_SUFFIXES = [
    ".h5",
    ".h5ad",
    ".mtx",
    ".mtx.gz",
    ".csv",
    ".csv.gz",
    ".tsv",
    ".tsv.gz",
    ".txt",
    ".txt.gz",
    ".tar",
    ".tar.gz",
    ".tgz",
    ".zip",
    ".json",
    ".parquet",
    ".png",
    ".jpg",
    ".jpeg",
    ".tif",
    ".tiff"
]


inventory_rows = []


for root, dirs, files in os.walk(
    TCGA_ROOT
):

    root_path = Path(root)

    # Do not recursively audit the newly created Step32 output itself
    if STEP32_ROOT in [
        root_path,
        *root_path.parents
    ]:
        continue

    for fname in files:

        fpath = root_path / fname

        lower_name = fname.lower()
        lower_path = str(
            fpath
        ).lower()


        keyword_hits = [
            k
            for k in SPATIAL_KEYWORDS
            if (
                k in lower_name
                or
                k in lower_path
            )
        ]


        suffix_match = any(
            lower_name.endswith(s)
            for s in RELEVANT_SUFFIXES
        )


        # Also preserve anything containing a GEO accession
        geo_hits = re.findall(
            r"GSE\d+|GSM\d+",
            str(fpath),
            flags=re.IGNORECASE
        )


        if (
            keyword_hits
            or
            geo_hits
        ) and suffix_match:

            try:
                size_mb = (
                    fpath.stat().st_size
                    /
                    1024**2
                )
            except Exception:
                size_mb = np.nan


            inventory_rows.append({

                "path":
                    str(fpath),

                "file_name":
                    fname,

                "parent":
                    str(root_path),

                "size_mb":
                    size_mb,

                "keyword_hits":
                    ";".join(
                        keyword_hits
                    ),

                "geo_accessions":
                    ";".join(
                        sorted(
                            set(
                                x.upper()
                                for x in geo_hits
                            )
                        )
                    )
            })


spatial_candidate_inventory = (
    pd.DataFrame(
        inventory_rows
    )
)


if len(
    spatial_candidate_inventory
) > 0:

    spatial_candidate_inventory = (
        spatial_candidate_inventory
        .sort_values(
            [
                "parent",
                "file_name"
            ]
        )
        .reset_index(
            drop=True
        )
    )


print(
    "Candidate spatial-related files:",
    len(
        spatial_candidate_inventory
    )
)


display(
    spatial_candidate_inventory.head(
        100
    )
)

In [ ]:
# =============================================================================
# Step32A Cell 246
# Detect characteristic spatial-platform file structures
# =============================================================================

platform_signal_rows = []


for root, dirs, files in os.walk(
    TCGA_ROOT
):

    root_path = Path(root)

    if STEP32_ROOT in [
        root_path,
        *root_path.parents
    ]:
        continue

    file_set = set(
        files
    )

    lower_files = {
        f.lower()
        for f in files
    }


    signals = []


    # ---------------------------------------------------------
    # 10x Visium / Space Ranger signals
    # ---------------------------------------------------------

    if (
        "filtered_feature_bc_matrix.h5"
        in lower_files
    ):
        signals.append(
            "10x_matrix_h5"
        )


    if any(
        "tissue_positions"
        in f
        for f in lower_files
    ):
        signals.append(
            "visium_tissue_positions"
        )


    if (
        "scalefactors_json.json"
        in lower_files
    ):
        signals.append(
            "visium_scalefactors"
        )


    if any(
        "tissue_hires_image"
        in f
        for f in lower_files
    ):
        signals.append(
            "visium_hires_image"
        )


    if any(
        "tissue_lowres_image"
        in f
        for f in lower_files
    ):
        signals.append(
            "visium_lowres_image"
        )


    # ---------------------------------------------------------
    # Matrix Market
    # ---------------------------------------------------------

    has_mtx = any(
        f.endswith(
            (
                ".mtx",
                ".mtx.gz"
            )
        )
        for f in lower_files
    )


    has_barcodes = any(
        "barcode"
        in f
        for f in lower_files
    )


    has_features = any(
        (
            "feature"
            in f
            or
            "gene"
            in f
        )
        for f in lower_files
    )


    if (
        has_mtx
        and
        has_barcodes
        and
        has_features
    ):
        signals.append(
            "matrix_market_expression"
        )


    # ---------------------------------------------------------
    # AnnData
    # ---------------------------------------------------------

    if any(
        f.endswith(
            ".h5ad"
        )
        for f in lower_files
    ):
        signals.append(
            "h5ad"
        )


    # ---------------------------------------------------------
    # Imaging
    # ---------------------------------------------------------

    if any(
        f.endswith(
            (
                ".tif",
                ".tiff",
                ".png",
                ".jpg",
                ".jpeg"
            )
        )
        for f in lower_files
    ):
        signals.append(
            "image_files"
        )


    if len(
        signals
    ) > 0:

        platform_signal_rows.append({

            "directory":
                str(root_path),

            "n_files":
                len(files),

            "signals":
                ";".join(
                    signals
                )
        })


platform_signal_table = (
    pd.DataFrame(
        platform_signal_rows
    )
)


if len(
    platform_signal_table
) > 0:

    platform_signal_table = (
        platform_signal_table
        .sort_values(
            "directory"
        )
        .reset_index(
            drop=True
        )
    )


print(
    "Directories with spatial/platform signals:",
    len(
        platform_signal_table
    )
)


display(
    platform_signal_table.head(
        100
    )
)

In [ ]:
# =============================================================================
# Step32A Cell 247
# GEO accession audit
# =============================================================================

geo_pattern = re.compile(
    r"\b(GSE\d+|GSM\d+)\b",
    flags=re.IGNORECASE
)


geo_rows = []


for root, dirs, files in os.walk(
    TCGA_ROOT
):

    root_path = Path(root)

    if STEP32_ROOT in [
        root_path,
        *root_path.parents
    ]:
        continue


    full_text = " ".join(
        [
            str(root_path),
            *files
        ]
    )


    accessions = sorted(
        set(
            x.upper()
            for x in geo_pattern.findall(
                full_text
            )
        )
    )


    if len(
        accessions
    ) > 0:

        geo_rows.append({

            "directory":
                str(root_path),

            "accessions":
                ";".join(
                    accessions
                ),

            "n_files":
                len(files)
        })


geo_inventory = (
    pd.DataFrame(
        geo_rows
    )
)


if len(
    geo_inventory
) > 0:

    geo_inventory = (
        geo_inventory
        .drop_duplicates()
        .sort_values(
            "directory"
        )
        .reset_index(
            drop=True
        )
    )


display(
    geo_inventory.head(
        100
    )
)

In [ ]:
# =============================================================================
# Step32A Cell 248
# Archive audit — no extraction
# =============================================================================

archive_rows = []


archive_suffixes = (
    ".tar",
    ".tar.gz",
    ".tgz",
    ".zip"
)


for root, dirs, files in os.walk(
    TCGA_ROOT
):

    root_path = Path(root)

    if STEP32_ROOT in [
        root_path,
        *root_path.parents
    ]:
        continue


    for fname in files:

        lower = fname.lower()

        if not lower.endswith(
            archive_suffixes
        ):
            continue


        fpath = root_path / fname

        spatial_keyword = any(
            k in str(
                fpath
            ).lower()
            for k in SPATIAL_KEYWORDS
        )


        geo_accessions = sorted(
            set(
                x.upper()
                for x in geo_pattern.findall(
                    str(fpath)
                )
            )
        )


        if (
            spatial_keyword
            or
            len(
                geo_accessions
            ) > 0
        ):

            archive_rows.append({

                "archive":
                    str(fpath),

                "geo_accessions":
                    ";".join(
                        geo_accessions
                    ),

                "size_mb":
                    fpath.stat().st_size
                    /
                    1024**2
            })


archive_inventory = (
    pd.DataFrame(
        archive_rows
    )
)


display(
    archive_inventory
)

In [ ]:
# =============================================================================
# Step32A Cell 249
# Conservative audit classification
# =============================================================================

n_candidate_files = len(
    spatial_candidate_inventory
)

n_signal_dirs = len(
    platform_signal_table
)

n_archives = len(
    archive_inventory
)


visium_signal = False


if n_signal_dirs > 0:

    visium_signal = (
        platform_signal_table[
            "signals"
        ]
        .str.contains(
            "visium_",
            regex=False
        )
        .any()
    )


if n_candidate_files == 0:

    audit_status = (
        "NO_LOCAL_SPATIAL_DATA_IDENTIFIED"
    )

elif visium_signal:

    audit_status = (
        "LOCAL_VISIUM_LIKE_DATA_IDENTIFIED"
    )

else:

    audit_status = (
        "LOCAL_SPATIAL_CANDIDATES_IDENTIFIED_PLATFORM_NOT_YET_LOCKED"
    )


print(
    "STEP32A audit status:",
    audit_status
)

print(
    "Candidate files:",
    n_candidate_files
)

print(
    "Platform-signal directories:",
    n_signal_dirs
)

print(
    "Candidate archives:",
    n_archives
)

print(
    "Visium-like signal:",
    visium_signal
)

In [ ]:
# =============================================================================
# Step32A Cell 250
# Save audit tables
# =============================================================================

SPATIAL_INVENTORY_PATH = (
    AUDIT_DIR /
    "Step32A_SpatialCandidateInventory.csv"
)

PLATFORM_SIGNAL_PATH = (
    AUDIT_DIR /
    "Step32A_PlatformSignalDirectories.csv"
)

GEO_INVENTORY_PATH = (
    AUDIT_DIR /
    "Step32A_GEOInventory.csv"
)

ARCHIVE_INVENTORY_PATH = (
    AUDIT_DIR /
    "Step32A_ArchiveInventory.csv"
)


spatial_candidate_inventory.to_csv(
    SPATIAL_INVENTORY_PATH,
    index=False
)

platform_signal_table.to_csv(
    PLATFORM_SIGNAL_PATH,
    index=False
)

geo_inventory.to_csv(
    GEO_INVENTORY_PATH,
    index=False
)

archive_inventory.to_csv(
    ARCHIVE_INVENTORY_PATH,
    index=False
)


step32a_audit = {

    "status":
        audit_status,

    "candidate_files":
        int(
            n_candidate_files
        ),

    "platform_signal_directories":
        int(
            n_signal_dirs
        ),

    "candidate_archives":
        int(
            n_archives
        ),

    "visium_like_signal":
        bool(
            visium_signal
        ),

    "new_packages_installed":
        False,

    "files_modified":
        False
}


STEP32A_JSON_PATH = (
    AUDIT_DIR /
    "Step32A_DATA_AUDIT.json"
)


with open(
    STEP32A_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step32a_audit,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    "Saved:",
    STEP32A_JSON_PATH
)

In [ ]:
# =============================================================================
# Step32A Cell 251
# REPORT
# =============================================================================

print("=" * 120)

print(
    "STEP32A — SPATIAL DATA / PLATFORM AUDIT"
)

print("=" * 120)


print(
    "\n[1] ENVIRONMENT"
)

print(
    "Python executable:",
    sys.executable
)

print(
    "Python version:",
    sys.version.split()[0]
)

print(
    "Scanpy:",
    None if sc is None else sc.__version__
)

print(
    "AnnData:",
    None if ad is None else ad.__version__
)


print(
    "\n[2] AUDIT STATUS"
)

print(
    audit_status
)


print(
    "\n[3] COUNTS"
)

print(
    "Candidate spatial-related files:",
    n_candidate_files
)

print(
    "Directories with platform signals:",
    n_signal_dirs
)

print(
    "Candidate archives:",
    n_archives
)

print(
    "Visium-like signal:",
    visium_signal
)


print(
    "\n[4] PLATFORM-SIGNAL DIRECTORIES"
)

if len(
    platform_signal_table
) == 0:

    print(
        "None identified."
    )

else:

    display(
        platform_signal_table.head(
            30
        )
    )


print(
    "\n[5] GEO ACCESSIONS / DIRECTORIES"
)

if len(
    geo_inventory
) == 0:

    print(
        "None identified."
    )

else:

    display(
        geo_inventory.head(
            30
        )
    )


print(
    "\n[6] CANDIDATE ARCHIVES"
)

if len(
    archive_inventory
) == 0:

    print(
        "None identified."
    )

else:

    display(
        archive_inventory.head(
            30
        )
    )


print(
    "\n[7] TOP CANDIDATE FILES"
)

if len(
    spatial_candidate_inventory
) == 0:

    print(
        "None identified."
    )

else:

    display(
        spatial_candidate_inventory[
            [
                "path",
                "size_mb",
                "keyword_hits",
                "geo_accessions"
            ]
        ].head(
            50
        )
    )


print(
    "\n[8] FIREWALL"
)

print(
    "Spatial analysis performed:",
    False
)

print(
    "Core50 reselection performed:",
    False
)

print(
    "New malignant states defined:",
    False
)

print(
    "New spatial packages installed:",
    False
)


print(
    "\nSTEP32A: COMPLETE"
)

print(
    "NEXT: determine spatial dataset/platform and lock Step32B input."
)

print("=" * 120)

## Step32B — Primary spatial dataset lock and archive audit

### Frozen primary dataset

GSE281759

Rationale:
- Human hepatocellular carcinoma
- Multi-region spatial transcriptomics
- 10x Genomics Visium
- Surgical-resection cohort
- Designed around HCC recurrence
- Multiple spatial regions across multiple patients

### Step32 role

GSE281759 will be used to spatially validate the frozen malignant-cell
framework established in Steps 23–31.

Primary target:
- MS3 Core50 / proliferative-mitotic high-risk biology

Secondary target:
- MS6 hepatic-differentiated/metabolic biology
- reciprocal MS3-like versus MS6-like spatial organization

No signatures or malignant states will be reselected.

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step32B Cell 11
# Locate the local GSE281759 archive
# =============================================================================

from pathlib import Path

TCGA_ROOT = Path(_hcc_path(''))


gse281759_candidates = sorted(
    [
        p
        for p in TCGA_ROOT.rglob("*GSE281759*")
        if p.is_file()
    ]
)


print(
    "GSE281759 candidate files:",
    len(gse281759_candidates)
)


for p in gse281759_candidates:

    print(
        f"{p} | "
        f"{p.stat().st_size / 1024**2:.2f} MB"
    )


archive_candidates = [
    p
    for p in gse281759_candidates
    if (
        p.name.lower().endswith(".tar")
        or
        p.name.lower().endswith(".tar.gz")
        or
        p.name.lower().endswith(".tgz")
    )
]


assert len(archive_candidates) >= 1, (
    "No local GSE281759 archive found."
)


# Prefer the largest archive if duplicate copies exist
GSE281759_ARCHIVE = max(
    archive_candidates,
    key=lambda p: p.stat().st_size
)


print(
    "\nPRIMARY LOCAL ARCHIVE:"
)

print(
    GSE281759_ARCHIVE
)

print(
    "Size MB:",
    GSE281759_ARCHIVE.stat().st_size / 1024**2
)

In [ ]:
# =============================================================================
# Step32B Cell 12
# Read-only archive member audit
# =============================================================================

import tarfile
import pandas as pd
import re


with tarfile.open(
    GSE281759_ARCHIVE,
    mode="r:*"
) as tar:

    members = [
        m
        for m in tar.getmembers()
        if m.isfile()
    ]


archive_member_rows = []


for m in members:

    name = Path(
        m.name
    ).name


    gsm_match = re.search(
        r"(GSM\d+)",
        name,
        flags=re.IGNORECASE
    )


    sample_match = re.search(
        r"GSM\d+_([^_]+)",
        name,
        flags=re.IGNORECASE
    )


    archive_member_rows.append({

        "member":
            m.name,

        "file_name":
            name,

        "gsm":
            (
                gsm_match.group(1).upper()
                if gsm_match
                else None
            ),

        "sample_label":
            (
                sample_match.group(1)
                if sample_match
                else None
            ),

        "size_mb":
            m.size / 1024**2
    })


archive_members = pd.DataFrame(
    archive_member_rows
)


print(
    "Files inside archive:",
    len(
        archive_members
    )
)


display(
    archive_members.head(
        50
    )
)

In [ ]:
# =============================================================================
# Step32B Cell 13
# Identify Visium samples and required file components
# =============================================================================

required_patterns = {

    "matrix":
        "matrix.mtx",

    "barcodes":
        "barcodes.tsv",

    "features":
        "features.tsv",

    "positions":
        "tissue_positions",

    "scalefactors":
        "scalefactors",

    "hires_image":
        "tissue_hires_image",

    "lowres_image":
        "tissue_lowres_image"
}


gsm_ids = sorted(
    archive_members[
        "gsm"
    ]
    .dropna()
    .unique()
)


sample_audit_rows = []


for gsm in gsm_ids:

    sub = archive_members.loc[
        archive_members[
            "gsm"
        ]
        ==
        gsm
    ]


    names = (
        sub[
            "file_name"
        ]
        .str.lower()
        .tolist()
    )


    row = {
        "gsm":
            gsm,

        "n_files":
            len(sub)
    }


    for key, pattern in required_patterns.items():

        row[key] = any(
            pattern in x
            for x in names
        )


    row[
        "core_expression_complete"
    ] = all(
        row[k]
        for k in [
            "matrix",
            "barcodes",
            "features"
        ]
    )


    row[
        "spatial_metadata_complete"
    ] = all(
        row[k]
        for k in [
            "positions",
            "scalefactors"
        ]
    )


    row[
        "image_available"
    ] = (
        row[
            "hires_image"
        ]
        or
        row[
            "lowres_image"
        ]
    )


    sample_audit_rows.append(
        row
    )


gse281759_sample_audit = (
    pd.DataFrame(
        sample_audit_rows
    )
)


display(
    gse281759_sample_audit
)


print(
    "\nGSM samples detected:",
    len(
        gse281759_sample_audit
    )
)


print(
    "Expression-complete:",
    int(
        gse281759_sample_audit[
            "core_expression_complete"
        ].sum()
    )
)


print(
    "Spatial-metadata-complete:",
    int(
        gse281759_sample_audit[
            "spatial_metadata_complete"
        ].sum()
    )
)


print(
    "With tissue image:",
    int(
        gse281759_sample_audit[
            "image_available"
        ].sum()
    )
)

In [ ]:
# =============================================================================
# Step32B Cell 14
# REPORT — GSE281759 input lock
# =============================================================================

print("=" * 120)

print(
    "STEP32B — GSE281759 PRIMARY SPATIAL INPUT AUDIT"
)

print("=" * 120)


print(
    "\n[1] PRIMARY DATASET"
)

print(
    "GEO:",
    "GSE281759"
)

print(
    "Platform:",
    "10x Genomics Visium"
)

print(
    "Role:",
    "Primary spatial validation cohort"
)


print(
    "\n[2] LOCAL ARCHIVE"
)

print(
    GSE281759_ARCHIVE
)

print(
    "Size MB:",
    round(
        GSE281759_ARCHIVE.stat().st_size / 1024**2,
        2
    )
)


print(
    "\n[3] ARCHIVE CONTENT"
)

print(
    "Files:",
    len(
        archive_members
    )
)

print(
    "GSM samples detected:",
    len(
        gse281759_sample_audit
    )
)


print(
    "\n[4] VISIUM COMPLETENESS"
)

print(
    "Expression-complete:",
    int(
        gse281759_sample_audit[
            "core_expression_complete"
        ].sum()
    ),
    "/",
    len(
        gse281759_sample_audit
    )
)

print(
    "Spatial-metadata-complete:",
    int(
        gse281759_sample_audit[
            "spatial_metadata_complete"
        ].sum()
    ),
    "/",
    len(
        gse281759_sample_audit
    )
)

print(
    "Image available:",
    int(
        gse281759_sample_audit[
            "image_available"
        ].sum()
    ),
    "/",
    len(
        gse281759_sample_audit
    )
)


print(
    "\n[5] SAMPLE TABLE"
)

display(
    gse281759_sample_audit
)


print(
    "\n[6] FIREWALL"
)

print(
    "Archive extracted:",
    False
)

print(
    "Spatial expression analyzed:",
    False
)

print(
    "Core50 scoring performed:",
    False
)

print(
    "New packages installed:",
    False
)


print(
    "\nSTEP32B: COMPLETE"
)

print(
    "NEXT: lock sample structure, extract standardized Visium files, and begin Step32C."
)

print("=" * 120)

## Step32B — Primary spatial dataset lock and archive audit

### Frozen primary dataset

GSE281759

Rationale:
- Human hepatocellular carcinoma
- Multi-region spatial transcriptomics
- 10x Genomics Visium
- Surgical-resection cohort
- Designed around HCC recurrence
- Multiple spatial regions across multiple patients

### Step32 role

GSE281759 will be used to spatially validate the frozen malignant-cell
framework established in Steps 23–31.

Primary target:
- MS3 Core50 / proliferative-mitotic high-risk biology

Secondary target:
- MS6 hepatic-differentiated/metabolic biology
- reciprocal MS3-like versus MS6-like spatial organization

No signatures or malignant states will be reselected.

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step32B Cell 11
# Locate the local GSE281759 archive
# =============================================================================

from pathlib import Path

TCGA_ROOT = Path(_hcc_path(''))


gse281759_candidates = sorted(
    [
        p
        for p in TCGA_ROOT.rglob("*GSE281759*")
        if p.is_file()
    ]
)


print(
    "GSE281759 candidate files:",
    len(gse281759_candidates)
)


for p in gse281759_candidates:

    print(
        f"{p} | "
        f"{p.stat().st_size / 1024**2:.2f} MB"
    )


archive_candidates = [
    p
    for p in gse281759_candidates
    if (
        p.name.lower().endswith(".tar")
        or
        p.name.lower().endswith(".tar.gz")
        or
        p.name.lower().endswith(".tgz")
    )
]


assert len(archive_candidates) >= 1, (
    "No local GSE281759 archive found."
)


# Prefer the largest archive if duplicate copies exist
GSE281759_ARCHIVE = max(
    archive_candidates,
    key=lambda p: p.stat().st_size
)


print(
    "\nPRIMARY LOCAL ARCHIVE:"
)

print(
    GSE281759_ARCHIVE
)

print(
    "Size MB:",
    GSE281759_ARCHIVE.stat().st_size / 1024**2
)

In [ ]:
# =============================================================================
# Step32B Cell 12
# Read-only archive member audit
# =============================================================================

import tarfile
import pandas as pd
import re


with tarfile.open(
    GSE281759_ARCHIVE,
    mode="r:*"
) as tar:

    members = [
        m
        for m in tar.getmembers()
        if m.isfile()
    ]


archive_member_rows = []


for m in members:

    name = Path(
        m.name
    ).name


    gsm_match = re.search(
        r"(GSM\d+)",
        name,
        flags=re.IGNORECASE
    )


    sample_match = re.search(
        r"GSM\d+_([^_]+)",
        name,
        flags=re.IGNORECASE
    )


    archive_member_rows.append({

        "member":
            m.name,

        "file_name":
            name,

        "gsm":
            (
                gsm_match.group(1).upper()
                if gsm_match
                else None
            ),

        "sample_label":
            (
                sample_match.group(1)
                if sample_match
                else None
            ),

        "size_mb":
            m.size / 1024**2
    })


archive_members = pd.DataFrame(
    archive_member_rows
)


print(
    "Files inside archive:",
    len(
        archive_members
    )
)


display(
    archive_members.head(
        50
    )
)

In [ ]:
# =============================================================================
# Step32B Cell 13
# Identify Visium samples and required file components
# =============================================================================

required_patterns = {

    "matrix":
        "matrix.mtx",

    "barcodes":
        "barcodes.tsv",

    "features":
        "features.tsv",

    "positions":
        "tissue_positions",

    "scalefactors":
        "scalefactors",

    "hires_image":
        "tissue_hires_image",

    "lowres_image":
        "tissue_lowres_image"
}


gsm_ids = sorted(
    archive_members[
        "gsm"
    ]
    .dropna()
    .unique()
)


sample_audit_rows = []


for gsm in gsm_ids:

    sub = archive_members.loc[
        archive_members[
            "gsm"
        ]
        ==
        gsm
    ]


    names = (
        sub[
            "file_name"
        ]
        .str.lower()
        .tolist()
    )


    row = {
        "gsm":
            gsm,

        "n_files":
            len(sub)
    }


    for key, pattern in required_patterns.items():

        row[key] = any(
            pattern in x
            for x in names
        )


    row[
        "core_expression_complete"
    ] = all(
        row[k]
        for k in [
            "matrix",
            "barcodes",
            "features"
        ]
    )


    row[
        "spatial_metadata_complete"
    ] = all(
        row[k]
        for k in [
            "positions",
            "scalefactors"
        ]
    )


    row[
        "image_available"
    ] = (
        row[
            "hires_image"
        ]
        or
        row[
            "lowres_image"
        ]
    )


    sample_audit_rows.append(
        row
    )


gse281759_sample_audit = (
    pd.DataFrame(
        sample_audit_rows
    )
)


display(
    gse281759_sample_audit
)


print(
    "\nGSM samples detected:",
    len(
        gse281759_sample_audit
    )
)


print(
    "Expression-complete:",
    int(
        gse281759_sample_audit[
            "core_expression_complete"
        ].sum()
    )
)


print(
    "Spatial-metadata-complete:",
    int(
        gse281759_sample_audit[
            "spatial_metadata_complete"
        ].sum()
    )
)


print(
    "With tissue image:",
    int(
        gse281759_sample_audit[
            "image_available"
        ].sum()
    )
)

In [ ]:
# =============================================================================
# Step32B Cell 14
# REPORT — GSE281759 input lock
# =============================================================================

print("=" * 120)

print(
    "STEP32B — GSE281759 PRIMARY SPATIAL INPUT AUDIT"
)

print("=" * 120)


print(
    "\n[1] PRIMARY DATASET"
)

print(
    "GEO:",
    "GSE281759"
)

print(
    "Platform:",
    "10x Genomics Visium"
)

print(
    "Role:",
    "Primary spatial validation cohort"
)


print(
    "\n[2] LOCAL ARCHIVE"
)

print(
    GSE281759_ARCHIVE
)

print(
    "Size MB:",
    round(
        GSE281759_ARCHIVE.stat().st_size / 1024**2,
        2
    )
)


print(
    "\n[3] ARCHIVE CONTENT"
)

print(
    "Files:",
    len(
        archive_members
    )
)

print(
    "GSM samples detected:",
    len(
        gse281759_sample_audit
    )
)


print(
    "\n[4] VISIUM COMPLETENESS"
)

print(
    "Expression-complete:",
    int(
        gse281759_sample_audit[
            "core_expression_complete"
        ].sum()
    ),
    "/",
    len(
        gse281759_sample_audit
    )
)

print(
    "Spatial-metadata-complete:",
    int(
        gse281759_sample_audit[
            "spatial_metadata_complete"
        ].sum()
    ),
    "/",
    len(
        gse281759_sample_audit
    )
)

print(
    "Image available:",
    int(
        gse281759_sample_audit[
            "image_available"
        ].sum()
    ),
    "/",
    len(
        gse281759_sample_audit
    )
)


print(
    "\n[5] SAMPLE TABLE"
)

display(
    gse281759_sample_audit
)


print(
    "\n[6] FIREWALL"
)

print(
    "Archive extracted:",
    False
)

print(
    "Spatial expression analyzed:",
    False
)

print(
    "Core50 scoring performed:",
    False
)

print(
    "New packages installed:",
    False
)


print(
    "\nSTEP32B: COMPLETE"
)

print(
    "NEXT: lock sample structure, extract standardized Visium files, and begin Step32C."
)

print("=" * 120)

## Step32C1 — Patient / section metadata lock

### Objective

Lock the hierarchical sample structure of GSE281759 before any spatial
expression analysis.

Hierarchy:

Patient
→ spatial tissue section
→ GSM sample
→ Visium spots

### Rules

- The 17 spatial sections are NOT treated as 17 independent patients.
- Patient identity is derived from the submitted sample labels.
- GEO metadata are retrieved from the official GEO record.
- Region/section codes are preserved exactly as submitted.
- No biological meaning is inferred from labels such as T1, T1R, T2 or T3
  unless explicitly supported by source metadata.
- No expression analysis or Core50 scoring is performed.

In [ ]:
# =============================================================================
# Step32C1 Cell 16
# Lock local GSM -> submitted sample-label mapping
# =============================================================================

import pandas as pd
import numpy as np
import re


assert "archive_members" in globals(), (
    "archive_members not found. Run Step32B first."
)


local_label_rows = []


for gsm, sub in archive_members.groupby(
    "gsm",
    observed=True
):

    labels = sorted(
        set(
            sub["sample_label"]
            .dropna()
            .astype(str)
        )
    )


    local_label_rows.append({
        "gsm":
            gsm,

        "n_archive_files":
            len(sub),

        "n_sample_labels":
            len(labels),

        "sample_label":
            labels[0]
            if len(labels) == 1
            else "|".join(labels)
    })


local_sample_map = (
    pd.DataFrame(
        local_label_rows
    )
    .sort_values("gsm")
    .reset_index(drop=True)
)


display(
    local_sample_map
)


assert len(local_sample_map) == 17

assert (
    local_sample_map[
        "n_sample_labels"
    ]
    ==
    1
).all()


print(
    "\nLocal GSM/sample-label mapping: PASS"
)

In [ ]:
# =============================================================================
# Step32C1 Cell 17
# Retrieve official GEO sample metadata
# =============================================================================

from urllib.request import Request, urlopen
from urllib.parse import urlencode
import time


def fetch_geo_text(url, timeout=30):

    req = Request(
        url,
        headers={
            "User-Agent":
                "Mozilla/5.0 Step32-GEO-metadata-audit"
        }
    )

    with urlopen(
        req,
        timeout=timeout
    ) as r:

        return r.read().decode(
            "utf-8",
            errors="replace"
        )


# -------------------------------------------------------------------------
# First attempt: retrieve all GSM records belonging to GSE281759
# -------------------------------------------------------------------------

params = {
    "acc":
        "GSE281759",

    "targ":
        "gsm",

    "view":
        "brief",

    "form":
        "text"
}


geo_url = (
    "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?"
    +
    urlencode(params)
)


geo_text = None
geo_fetch_mode = None


try:

    geo_text = fetch_geo_text(
        geo_url
    )

    if "^SAMPLE" in geo_text:

        geo_fetch_mode = (
            "GSE281759_GSM_BATCH"
        )

except Exception as e:

    print(
        "Batch GEO retrieval failed:",
        repr(e)
    )


print(
    "GEO fetch mode:",
    geo_fetch_mode
)

print(
    "Characters retrieved:",
    0 if geo_text is None else len(geo_text)
)

In [ ]:
# =============================================================================
# Step32C1 Cell 18
# Parse official GEO metadata and construct patient hierarchy
# =============================================================================

assert geo_text is not None
assert "^SAMPLE" in geo_text


def parse_geo_sample_blocks(text):

    blocks = re.split(
        r"(?=\^SAMPLE\s*=\s*GSM\d+)",
        text
    )

    rows = []


    for block in blocks:

        gsm_match = re.search(
            r"\^SAMPLE\s*=\s*(GSM\d+)",
            block
        )

        if not gsm_match:
            continue


        gsm = gsm_match.group(1)


        def first_value(key):

            m = re.search(
                rf"^!{re.escape(key)}\s*=\s*(.*)$",
                block,
                flags=re.MULTILINE
            )

            return (
                m.group(1).strip()
                if m
                else None
            )


        char_lines = re.findall(
            r"^!Sample_characteristics_ch1\s*=\s*(.*)$",
            block,
            flags=re.MULTILINE
        )


        characteristics = {}


        for x in char_lines:

            if ":" in x:

                key, value = x.split(
                    ":",
                    1
                )

                characteristics[
                    key.strip().lower()
                ] = value.strip()


        rows.append({

            "gsm":
                gsm,

            "geo_title":
                first_value(
                    "Sample_title"
                ),

            "source_name":
                first_value(
                    "Sample_source_name_ch1"
                ),

            "description":
                first_value(
                    "Sample_description"
                ),

            "tissue":
                characteristics.get(
                    "tissue"
                ),

            "sex":
                characteristics.get(
                    "sex"
                ),

            "tumour_stage":
                (
                    characteristics.get(
                        "tumour stage"
                    )
                    or
                    characteristics.get(
                        "tumor stage"
                    )
                )
        })


    return pd.DataFrame(
        rows
    )


geo_sample_metadata = (
    parse_geo_sample_blocks(
        geo_text
    )
)


print(
    "GEO samples parsed:",
    len(
        geo_sample_metadata
    )
)


display(
    geo_sample_metadata
)

In [ ]:
# =============================================================================
# Step32C1 Cell 19A
# Diagnose sample-label parsing failure
# =============================================================================

diagnostic = (
    step32_sample_metadata[
        [
            "gsm",
            "sample_label",
            "geo_title",
            "source_name",
            "sex",
            "tumour_stage"
        ]
    ]
    .copy()
)


old_parse = (
    diagnostic[
        "sample_label"
    ]
    .str.extract(
        r"^(?P<patient_id>.+?)(?P<section_code>T\d.*)$"
    )
)


diagnostic[
    "old_patient_id"
] = old_parse[
    "patient_id"
]


diagnostic[
    "old_section_code"
] = old_parse[
    "section_code"
]


print(
    "All submitted sample labels:"
)

display(
    diagnostic
)


print(
    "\nFAILED UNDER OLD T-pattern:"
)

display(
    diagnostic.loc[
        diagnostic[
            "old_patient_id"
        ].isna()
        |
        diagnostic[
            "old_section_code"
        ].isna()
    ]
)

In [ ]:
# =============================================================================
# Step32C1 Cell 19B
# Canonical patient-ID harmonization
# =============================================================================

import pandas as pd


# -------------------------------------------------------------------------
# Preserve raw parsed patient ID
# -------------------------------------------------------------------------

step32_sample_metadata[
    "raw_patient_id"
] = (
    step32_sample_metadata[
        "patient_id"
    ]
    .astype(str)
)


# -------------------------------------------------------------------------
# Explicit frozen alias map
#
# Source harmonization only:
# GSE281759 publication uses B013,
# while some GEO/filename records use B0013.
# -------------------------------------------------------------------------

PATIENT_ALIAS_MAP = {
    "B0013": "B013"
}


step32_sample_metadata[
    "canonical_patient_id"
] = (
    step32_sample_metadata[
        "raw_patient_id"
    ]
    .replace(
        PATIENT_ALIAS_MAP
    )
)


step32_sample_metadata[
    "patient_id_harmonized"
] = (
    step32_sample_metadata[
        "raw_patient_id"
    ]
    !=
    step32_sample_metadata[
        "canonical_patient_id"
    ]
)


# -------------------------------------------------------------------------
# Audit
# -------------------------------------------------------------------------

print(
    "RAW patient IDs:"
)

print(
    sorted(
        step32_sample_metadata[
            "raw_patient_id"
        ].unique()
    )
)


print(
    "\nCANONICAL patient IDs:"
)

print(
    sorted(
        step32_sample_metadata[
            "canonical_patient_id"
        ].unique()
    )
)


print(
    "\nHarmonized records:"
)

display(
    step32_sample_metadata.loc[
        step32_sample_metadata[
            "patient_id_harmonized"
        ],
        [
            "gsm",
            "sample_label",
            "raw_patient_id",
            "canonical_patient_id",
            "section_code",
            "geo_title"
        ]
    ]
)


# -------------------------------------------------------------------------
# Final patient count
# -------------------------------------------------------------------------

n_canonical_patients = (
    step32_sample_metadata[
        "canonical_patient_id"
    ]
    .nunique()
)


assert n_canonical_patients == 4, (
    f"Expected 4 canonical patients, "
    f"found {n_canonical_patients}"
)


EXPECTED_PATIENTS = {
    "A009",
    "B013",
    "C004",
    "C008"
}


assert set(
    step32_sample_metadata[
        "canonical_patient_id"
    ]
) == EXPECTED_PATIENTS


# -------------------------------------------------------------------------
# Final patient-section summary
# -------------------------------------------------------------------------

patient_section_summary = (
    step32_sample_metadata
    .groupby(
        [
            "canonical_patient_id",
            "sex",
            "tumour_stage"
        ],
        dropna=False,
        observed=True
    )
    .agg(

        n_spatial_sections=(
            "gsm",
            "nunique"
        ),

        section_codes=(
            "section_code",
            lambda x:
                ";".join(
                    sorted(
                        x.astype(str)
                    )
                )
        ),

        sample_labels=(
            "sample_label",
            lambda x:
                ";".join(
                    sorted(
                        x.astype(str)
                    )
                )
        ),

        gsm_ids=(
            "gsm",
            lambda x:
                ";".join(
                    sorted(
                        x.astype(str)
                    )
                )
        )
    )
    .reset_index()
)


print(
    "\nFINAL SECTION-LEVEL METADATA"
)

display(
    step32_sample_metadata[
        [
            "gsm",
            "sample_label",
            "raw_patient_id",
            "canonical_patient_id",
            "section_code",
            "sex",
            "tumour_stage",
            "patient_id_harmonized"
        ]
    ]
)


print(
    "\nFINAL PATIENT STRUCTURE"
)

display(
    patient_section_summary
)


print(
    "\nCanonical patients:",
    n_canonical_patients
)

print(
    "Spatial sections:",
    step32_sample_metadata[
        "gsm"
    ].nunique()
)


print(
    "\nSTEP32C1 PATIENT HIERARCHY: PASS"
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step32C1 Cell 20
# Audit previously prepared Step32 transcriptomics backup folder
# =============================================================================

from pathlib import Path
import pandas as pd
import hashlib


BACKUP_STEP32_DIR = Path(
    _hcc_path('LIHC_scRNA_Step32_Transcriptomics')
)


print(
    "Backup directory:"
)

print(
    BACKUP_STEP32_DIR
)

print(
    "Exists:",
    BACKUP_STEP32_DIR.exists()
)


assert BACKUP_STEP32_DIR.exists()


backup_rows = []


for p in sorted(
    BACKUP_STEP32_DIR.rglob("*")
):

    if not p.is_file():
        continue

    name_upper = p.name.upper()


    geo_hits = sorted(
        set(
            re.findall(
                r"GSE\d+|GSM\d+",
                str(p),
                flags=re.IGNORECASE
            )
        )
    )


    backup_rows.append({

        "path":
            str(p),

        "file_name":
            p.name,

        "size_mb":
            p.stat().st_size / 1024**2,

        "geo_accessions":
            ";".join(
                x.upper()
                for x in geo_hits
            ),

        "suffix":
            "".join(
                p.suffixes
            )
    })


backup_inventory = pd.DataFrame(
    backup_rows
)


print(
    "\nFiles:",
    len(
        backup_inventory
    )
)


display(
    backup_inventory[
        [
            "path",
            "size_mb",
            "geo_accessions",
            "suffix"
        ]
    ].head(
        100
    )
)


print(
    "\nGEO datasets represented:"
)


all_geo = sorted(
    set(
        x
        for cell in backup_inventory[
            "geo_accessions"
        ]
        for x in cell.split(";")
        if x
    )
)


print(
    all_geo
)

In [ ]:
# =============================================================================
# Step32C1 Cell 21
# Duplicate GSE281759 archive integrity audit
# =============================================================================

import hashlib


gse281759_local_archives = []


for base in [
    TCGA_ROOT,
    BACKUP_STEP32_DIR
]:

    for p in base.rglob("*GSE281759*"):

        if (
            p.is_file()
            and
            (
                p.name.lower().endswith(".tar")
                or
                p.name.lower().endswith(".tar.gz")
                or
                p.name.lower().endswith(".tgz")
            )
        ):

            gse281759_local_archives.append(
                p
            )


# Remove duplicate path records
gse281759_local_archives = sorted(
    set(
        gse281759_local_archives
    )
)


def sha256_file(
    path,
    chunk_size=1024 * 1024 * 8
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


archive_compare_rows = []


for p in gse281759_local_archives:

    print(
        "Hashing:",
        p
    )

    archive_compare_rows.append({

        "path":
            str(p),

        "size_mb":
            p.stat().st_size / 1024**2,

        "sha256":
            sha256_file(
                p
            )
    })


gse281759_archive_compare = pd.DataFrame(
    archive_compare_rows
)


display(
    gse281759_archive_compare
)


print(
    "\nUnique SHA256:",
    gse281759_archive_compare[
        "sha256"
    ].nunique()
)


if (
    gse281759_archive_compare[
        "sha256"
    ].nunique()
    ==
    1
):

    print(
        "GSE281759 duplicate archives are byte-identical: PASS"
    )

else:

    print(
        "WARNING: GSE281759 archives are not byte-identical."
    )

## Step32C2 — Standardized Visium extraction and read audit

Purpose: stage only the frozen GSE281759 tar, preserve all raw members, and verify 17 expression and spatial inputs without normalization. The audited script is rerunnable and skips existing files of matching size.

In [ ]:
from hcc_release_paths import project_path as _hcc_path
from pathlib import Path
import json, subprocess, sys
root = Path(_hcc_path('LIHC_scRNA_Step32_Spatial_Validation'))
c2_lock_path = root / 'Audit/Step32C2_FINAL_LOCK.json'
if not c2_lock_path.exists():
    subprocess.run([sys.executable, '-u', str(root / 'Audit/Step32C2_run.py')], check=True)
print('C2 lock:', c2_lock_path)
print(json.dumps(json.loads(c2_lock_path.read_text()), indent=2))

In [ ]:
import pandas as pd
c2 = json.loads(c2_lock_path.read_text())
expr = pd.read_csv(root / 'Audit/Step32C2_ExpressionReadAudit.csv')
spatial = pd.read_csv(root / 'Audit/Step32C2_SpatialReadAudit.csv')
assert c2['status'] == 'PASS' and len(expr) == len(spatial) == 17
display(expr[['gsm','n_spots','n_genes','nonnegative','integer_like_raw_counts']])
display(spatial[['gsm','n_overlap','overlap_fraction','barcode_alignment_ok']])

In [ ]:
# Step32C2 REPORT
print('Step32C2 PASS: 17/17 expression and spatial inputs readable; '      '17/17 nonnegative and integer-like; 17/17 barcode alignment OK.')
print('Canonical patient count:', c2['n_canonical_patients'])
print('No normalization, QC filtering, or scoring was performed in C2.')

## Step32D — Raw spatial QC

Purpose: describe raw total counts, detected genes, mitochondrial fraction, and tissue flags by section and canonical patient. The universe keeps every filtered-expression spot because all 27,224 are in tissue; no additional QC threshold is applied. Raw counts and coordinates are frozen per GSM in H5AD.

In [ ]:
d_lock_path = root / 'Audit/Step32D_FINAL_LOCK.json'
if not d_lock_path.exists():
    subprocess.run([sys.executable, '-u', str(root / 'Audit/Step32D_qc_audit.py')], check=True)
    subprocess.run([sys.executable, '-u', str(root / 'Audit/Step32D_freeze.py')], check=True)
d = json.loads(d_lock_path.read_text())
print({key: value for key, value in d.items() if key != 'h5ad_files'})

In [ ]:
section_qc = pd.read_csv(root / 'Tables/Step32D_QCSectionSummary.csv')
patient_qc = pd.read_csv(root / 'Tables/Step32D_QCCanonicalPatientSummary.csv')
display(section_qc.loc[section_qc.scope == 'in_tissue', ['gsm','n_spots','n_zero_count','n_under_200_genes','n_pct_mt_gt_20','total_counts_median','n_genes_by_counts_median','pct_counts_mt_median']])
display(patient_qc)

In [ ]:
# Step32D REPORT
assert d['status'] == 'PASS' and d['n_sections'] == 17
print('Step32D PASS: 27,224/27,224 expression spots retained; all in tissue.')
print('No additional QC threshold or spot removal; 17 raw-count H5AD files frozen.')
print('Descriptive flags only: 98 spots <200 genes; 82 spots >20% mitochondrial counts.')
print('Biological signatures were not used for QC.')

## Step32 gene identifier harmonization

Purpose: use the frozen Step25 Core50 membership and Step31 Ensembl mappings. No genes are selected or substituted. Verify coverage in all 17 spatial sections before scoring.

In [ ]:
from hcc_release_paths import project_path as _hcc_path
from pathlib import Path
import json, subprocess, sys
import pandas as pd
root = Path(_hcc_path('LIHC_scRNA_Step32_Spatial_Validation'))
coverage_path = root / 'Audit/Step32_GeneCoverageAudit.json'
if not coverage_path.exists():
    subprocess.run([sys.executable, str(root / 'Audit/Step32_GeneCoverageAudit.py')], check=True)
coverage = json.loads(coverage_path.read_text())
display(pd.read_csv(root / 'Audit/Step32_GeneProgramCoverageSummary.csv'))

In [ ]:
# Gene harmonization REPORT
assert coverage['core50_present'] == 50 and coverage['n_sections'] == 17
assert all(r['n_present'] == r['n_frozen_genes'] for r in coverage['programs'])
print('PASS: 36,601 shared spatial Ensembl IDs; Core50 50/50 and all five frozen programs fully mapped.')
print('Frozen SGO1/SGO2 aliases retained from Step31; Core50 membership unchanged.')

## Step32E — Primary spatial validation of frozen MS3 Core50

Purpose: score the frozen Core50 and three independent proliferative programs using Step31 deterministic gene-z-mean logic after 10,000-count log normalization. Spot associations are descriptive. Summaries retain section nesting and four independent canonical patients.

In [ ]:
ef_lock_path = root / 'Audit/Step32EF_FINAL_LOCK.json'
if not ef_lock_path.exists():
    subprocess.run([sys.executable, str(root / 'Audit/Step32EF_score.py')], check=True)
    subprocess.run([sys.executable, str(root / 'Audit/Step32EF_technical_sensitivity.py')], check=True)
ef = json.loads(ef_lock_path.read_text())
e_patient = pd.read_csv(root / 'Tables/Step32E_Core50ProgramByPatient.csv')
tech = pd.read_csv(root / 'Tables/Step32EF_TechnicalSensitivityBySection.csv')
display(e_patient)

In [ ]:
# Step32E REPORT
e_tech = tech[tech.comparison.str.startswith('Core50_vs_')]
assert ef['status'] == 'PASS' and len(e_tech) == 51
print('Core50 versus each of three independent programs: 17/17 sections positive.')
print('Technical-adjusted sensitivity: 51/51 section-program associations positive.')
print('Patient is the independent biological unit (n=4); spot correlations are descriptive.')

## Step32F — Secondary MS3-like versus MS6-like organization

Purpose: compare the three frozen independent proliferative programs with two frozen hepatic/metabolic programs, excluding Core50 from axis construction. Use full-resolution coordinates for a six-neighbor descriptive spatial check. Report inconsistent results without redefining a state or signature.

In [ ]:
f_section = pd.read_csv(root / 'Tables/Step32F_MS3MS6BySection.csv')
f_patient = pd.read_csv(root / 'Tables/Step32F_MS3MS6ByPatient.csv')
f_tech = tech[tech.comparison == 'MS3_like_vs_MS6_like']
display(f_patient)
display(f_section[['gsm','rho_within_spot_ms3_ms6','rho_ms3_neighbor_ms6']])

In [ ]:
# Step32F REPORT
print('Raw positive sections:', int((f_section.rho_within_spot_ms3_ms6 > 0).sum()), '/17')
print('Technical-adjusted negative sections:', int((f_tech.technical_partial_rho < 0).sum()), '/17')
print('Patient-level rho:', pd.read_csv(root / 'Tables/Step32EF_PatientLevelAssociation.csv').loc[lambda x: x.comparison == 'MS3_like_vs_MS6_like', 'patient_level_rho'].iloc[0])
print('REPORT: uniform reciprocal separation is not supported. No lineage or causal claim.')

## Step32 figures — bounded article evidence

Five single-panel figures show the three spatial score maps, primary section consistency, and the heterogeneous reciprocal-axis boundary. All figure text is English; PDF and 600 dpi PNG are exported.

In [ ]:
figure_paths = sorted((root / 'Figures').glob('Step32*.pdf'))
assert len(figure_paths) == 5
for p in figure_paths: print(p.name, 'PNG:', p.with_suffix('.png').exists())
figure_qa = json.loads((root / 'Audit/Step32_FigureQA.json').read_text())
print('PDF text audit passes:', sum(x['pdf_text_audit_exit_code'] == 0 for x in figure_qa['figure_checks']), '/5')

In [ ]:
# Figure REPORT
print('Five 600 dpi PNG/PDF figure pairs generated and visually inspected.')
print('Automated PDF collision audit unavailable: PyMuPDF absent; no package installed.')
print('Core50 spatial concordance is the primary result; reciprocal axis remains a limitation.')

## Step32 final report and claim boundary

Purpose: close the spatial validation with the frozen input, complete read/QC audits, primary positive result, secondary negative result, limitations, and output manifest. The article main conclusion is not rewritten.

In [ ]:
from hcc_release_paths import project_path as _hcc_path
from pathlib import Path
import json
root = Path(_hcc_path('LIHC_scRNA_Step32_Spatial_Validation'))
final_audit = json.loads((root / 'Audit/Step32_FinalAudit.json').read_text())
master = root / 'Step32_MASTER_REPORT.md'
manifest = root / 'Audit/Step32_OutputManifest.csv'
assert master.exists() and manifest.exists()
print('Master report:', master)
print('Output manifest:', manifest)
print('Final status:', final_audit['status'])

In [ ]:
# Step32 FINAL REPORT
assert final_audit['C2'] == final_audit['D'] == final_audit['E_F'] == 'PASS'
print('Primary: frozen Core50 aligns with independent proliferative/mitotic programs across sections and patients.')
print('Secondary: uniform MS3-like/MS6-like spatial reciprocity is not supported.')
print('Independent patients: 4; sections: 17; spots: 27,224.')
print('No causal, lineage-conversion, or biochemical activation claim.')

## Step32 reproducibility addendum

Purpose: verify that the executed analysis code is retained in the Notebook and copied to standalone scripts with explicit paths, parameters, package freeze, source hashes, and stage-to-cell index. Frozen analytical results are not rerun or overwritten.

In [ ]:
from hcc_release_paths import project_path as _hcc_path
from pathlib import Path
import json, hashlib
root = Path(_hcc_path('LIHC_scRNA_Step32_Spatial_Validation'))
audit = root / 'Audit'
rep = json.loads((audit / 'Step32_Reproducibility_Manifest.json').read_text())
env = (audit / 'Step32_environment_freeze.txt').read_text()
index = (audit / 'Step32_CODE_INDEX.md').read_text()
method = json.loads((audit / 'Step32EF_ScoringMethod.json').read_text())
assert len(rep['scripts']) == 11
assert all(x['executed_source_body_retained'] for x in rep['scripts'])
assert all(x in env for x in ['## python --version','## pip freeze','## conda list -n spatial32_wsl'])
assert rep['scoring']['target_sum'] == 10000 and rep['scoring']['z_ddof'] == 0
assert method['Core50_in_reciprocal_axis'] is False
assert rep['spatial']['neighbor_k'] == 6
assert '52-54' in index
print('Standalone scripts verified:', len(rep['scripts']))
print('Environment SHA256:', rep['environment_freeze_sha256'])

In [ ]:
# Step32 reproducibility REPORT
print('Notebook code retained: YES')
print('Standalone scripts retained: YES')
print('Environment frozen: YES')
print('Reproducibility manifest created: YES')
print('Frozen analysis outputs were not recomputed in this addendum.')

## Step32 final article-centered lock

**Primary.** Frozen MS3 Core50 has positive spatial concordance with G2M, mitotic-entry, and spindle/chromosome-segregation programs in all 17 Visium sections from four canonical patients. All 51 section-program associations remain positive after technical-complexity adjustment.

**Secondary negative finding.** The MS3-like versus MS6-like transcriptional relationship does not reproduce as a consistent spatially exclusive pattern: unadjusted same-spot associations are positive in 14/17 sections; adjusted directions are 8 positive and 9 negative. Do not call this spatial axis reciprocal or exclusive.

**Boundaries.** Sections are nested within four patients; spots are not independent biological replicates. Spatial concordance does not establish causality. The negative spatial result does not invalidate the prior transcriptional reciprocal relationship. No new spatial analysis was run for this lock.


In [ ]:
from hcc_release_paths import project_path as _hcc_path
# Recreate only the article-centered lock from existing Step32 audit outputs.
import runpy
runpy.run_path(_hcc_path('LIHC_scRNA_Step32_Spatial_Validation/Audit/Scripts/Step32_article_finalize.py'), run_name='__main__')
